In [1]:
from openai import OpenAI
from typing import Literal
from pathlib import Path
from pydantic import BaseModel
from dotenv import load_dotenv
import json
import os

Definido Modelo LLM Local

In [2]:
load_dotenv(dotenv_path=Path(".env"))

MODEL = "sabiazinho-4"

client = OpenAI(
    base_url="https://chat.maritaca.ai/api",
    api_key=os.getenv("MARITACA_API_KEY"),
)

In [3]:
class StepDecision(BaseModel):
    action: Literal["use_cag", "use_rag", "use_mongo", "final_answer"]
    reason: str
    query: str

In [4]:
def decide_next_step(question: str, context: list[dict]) -> StepDecision:
    prompt = f"""
Voce e um agente RAG agentico.

Sua tarefa NAO e responder diretamente.
Sua tarefa e escolher o proximo passo para responder a pergunta do usuario.

Acoes disponiveis:
- use_cag: consultar cache de conhecimento, regras gerais, politicas, FAQ e resumos.
- use_rag: buscar em documentos longos, PDFs, contratos, manuais e trechos especificos.
- use_mongo: consultar dados estruturados como clientes, pedidos, produtos e registros.
- final_answer: usar quando o contexto coletado ja for suficiente para responder.

Pergunta original:
{question}

Contexto coletado ate agora:
{json.dumps(context, ensure_ascii=False, indent=2)}

Responda APENAS JSON valido, sem markdown, neste formato:
{{
  "action": "use_cag",
  "reason": "motivo curto",
  "query": "consulta otimizada"
}}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    raw_text = response.choices[0].message.content
    json_start = raw_text.find("{")
    json_end = raw_text.rfind("}")

    if json_start == -1 or json_end == -1:
        raise ValueError(f"Resposta sem JSON valido: {raw_text}")

    data = json.loads(raw_text[json_start:json_end + 1])

    return StepDecision(**data)

In [5]:
def run_cag(query: str) -> str:
    cache_path = Path("cache.json")
    cache = json.loads(cache_path.read_text(encoding="utf-8"))

    prompt = f"""
Voce e um assistente CAG.

Responda usando APENAS o cache abaixo.
Se a informacao nao estiver no cache, diga que nao encontrou no cache.
Responda em portugues do Brasil.

Cache:
{json.dumps(cache, ensure_ascii=False, indent=2)}

Pergunta:
{query}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    return response.choices[0].message.content


def run_rag(query: str) -> str:
    return "RAG ainda nao implementado."

def run_mongo(query: str) -> str:
    return "Mongo ainda nao implementado."

In [7]:
def generate_final_answer(question: str, context: list[dict]) -> str:
    prompt = f"""
Voce deve responder a pergunta usando o contexto coletado.

Pergunta:
{question}

Contexto:
{json.dumps(context, ensure_ascii=False, indent=2)}

Regras:
- Responda em portugues do Brasil.
- Use apenas o contexto coletado.
- Se nao houver informacao suficiente, diga isso claramente.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    return response.choices[0].message.content


def agentic_loop(question: str) -> str:
    context = []
    seen_actions = set()

    for step in range(5):
        decision = decide_next_step(question, context)
        action_key = (decision.action, decision.query)

        if decision.action == "final_answer":
            return generate_final_answer(question, context)

        if action_key in seen_actions:
            return generate_final_answer(question, context)

        seen_actions.add(action_key)

        if decision.action == "use_cag":
            result = run_cag(decision.query)
        
        elif decision.action == "use_rag":
            result = run_rag(decision.query)

        elif decision.action == "use_mongo":
            result = run_mongo(decision.query)

        context.append({
            "step": step + 1,
            "action": decision.action,
            "query": decision.query,
            "reason": decision.reason,
            "result": result,
        })

    return generate_final_answer(question, context)